In [11]:
"""
SIR Monte Carlo: Original (brute-force) vs Cell-Listing
"""

import numpy as np
import matplotlib.pyplot as plt
import time

# =============================================================================
# 1. HealthStatus Function (initial conditions)

# Function to assign initial positions and health status
# Input
#  N: population size
#  L: length of square area
#  ni: size of initial infected population

#  Output
#  S: indicator for susceptible individuals
#  H: indicator for infected individuals
#  R: indicator for recovered individuals
#  sA: Social activity of a person
#  (x,y) : x,y coordinates of individuals 
# =============================================================================

def HealthStatus(N, L, ni, rng):
    x = rng.random(N) * L # x coordinate
    y = rng.random(N) * L # y coordinate
    
    S = np.ones(N)        # 1 - susceptible, 0 - not susceptible
    H = np.zeros(N)       # 1 - infected, 0 - not infected
    Hdays = np.zeros(N)   # Initialized to zero
    R = np.zeros(N)       # 1 -recovered, 0 - not recovered
    sA = np.ones(N)       # ????????????????????????????????????????????????????????????????????????????????????
    
    ind = rng.choice(N, size=ni, replace=False) # pick ni individuals randomly from N to be infected
    H[ind] = 1  # Set the selected individuals to infected
    S[ind] = 0  # So the individual is not susceptible
    return S, H, R, Hdays, sA, x, y

# =============================================================================
# 3. Update Positions
# ===========================================
def updatePositions(x, y, L, N, sig2, rng):
    """Compute displacements as a Brownian motion with periodic boundaries."""
    sig = np.sqrt(sig2)
    x = np.mod(x + rng.normal(0, sig, N), L)
    y = np.mod(y + rng.normal(0, sig, N), L)
    return x, y

# =============================================================================
# 4. Update Infection Days
# ===========================================
def updateInfectionDays(H, Hdays):
    """Find the infected individuals and increment their days of infection."""
    Hdays[H == 1] += 1
    return Hdays

# =============================================================================
# 5. Infected to Recovered
# ===========================================
def InfectedToRecovered(H, R, Hdays, rcd):
    """Converts infected individuals to recovered after rcd days of infection."""
    """Shared by BOTH methods"""
    ind = np.where(H == 1)[0] # Find the infected individuals
    rec = (Hdays[ind] == rcd).astype(int) # If infection days = rcd...
    R[ind] = rec    # ...set to recovered
    H[ind] -= rec   # The individual does not join the susceptible pool
    return H, R

In [13]:
# =============================================================================
# 6. Susceptible to infected (No Cell Listing - ORIGINAL METHOD)
# =============================================================================

def SusceptibleToInfected_Original(S, H, sA, x, y, sig2, rng):
    """
    For each susceptible i, compute sum_j sA[j]*exp(-r_ij^2 / 2sig2)
    """
    indi = np.where(S == 1)[0] # Find the suscectable individuals
    indj = np.where(H == 1)[0] # Find the infected individuals
    if indi.size == 0 or indj.size == 0:
        return S, H

    # shapes: (n_S, 1) - (1, n_I)  →  (n_S, n_I)
    # (n_S, n_I)
    dx = x[indi, None] - x[None, indj]  # find x-direction distance        
    dy = y[indi, None] - y[None, indj]  # find y-direction distance
    r2 = dx**2 + dy**2                           # (n_S, n_I)
    weights = sA[indj][None, :] * np.exp(-r2 / (2.0 * sig2))   # (n_S, n_I)

    #Update here
    #The original code uses for loop, but it's too slow in practice
    #This improvement method was suggested by AI to speed things up
    pi = weights.sum(axis=1)                     # (n_S,)

    rand_vals = rng.random(len(indi)) # draws one random number per susceptible individual
    newly_infected = indi[np.floor(pi + rand_vals).astype(int) > 0]
    H[newly_infected] = 1
    S[newly_infected] = 0
    return S, H

In [15]:
# =============================================================================
# 7. CELL-LISTING METHOD — S→I only
# =============================================================================

def build_cell_list(x, y, L, csl):
    
    num_cells = max(1, int(np.floor(L / csl))) # Update here
    # remove the if function
    # floor function handles the case L is not divided by csl 
    # max(1,) make sure that we have at least one cell, which is the entire area

    
    cell_size = L / num_cells
    cx = (np.floor(x / cell_size).astype(int)) % num_cells
    cy = (np.floor(y / cell_size).astype(int)) % num_cells
    cell_indices = cx * num_cells + cy          # flat index
    return cell_indices, num_cells, cell_size


def SusceptibleToInfected_CellList(S, H, sA, x, y, sig2, L, rng,
                                    cell_indices=None, num_cells=None, cell_size=None):
    """
    cell-listing S→I
    For each susceptible i, gather infected neighbours from the 3×3
    cell neighbourhood, then compute the kernel with NumPy ops.
    Building kernel with NumPy ops used a bit AI help, because people who are in charge of this section are not familiar with this skill.
    We are theoretical mathematicians, not that good with coding. 
    
    The minimum-image convention is applied for periodic boundaries (of the entire area boundary).
    """
    csl   = 3.0 * np.sqrt(sig2)
    csl2  = csl ** 2

    if cell_indices is None or num_cells is None or cell_size is None:
        cell_indices, num_cells, cell_size = build_cell_list(x, y, L, csl)

    # Pre-sort infected indices by cell for fast lookup
    indj_all = np.where(H == 1)[0]
    indi_all  = np.where(S == 1)[0]
    if indi_all.size == 0 or indj_all.size == 0:
        return S, H

    # Build a dict: flat_cell_id → array of infected indices in that cell
    inf_cell_ids = cell_indices[indj_all]
    # Use np.unique to group
    order = np.argsort(inf_cell_ids)
    sorted_ids   = inf_cell_ids[order]
    sorted_indj  = indj_all[order]
    unique_ids, first_occ = np.unique(sorted_ids, return_index=True)
    infected_by_cell = {}
    for k, uid in enumerate(unique_ids):
        start = first_occ[k]
        end   = first_occ[k+1] if k+1 < len(first_occ) else len(sorted_indj)
        infected_by_cell[uid] = sorted_indj[start:end]

    NC = num_cells
    pi_all   = np.zeros(len(indi_all), dtype=np.float64)

    for idx, i in enumerate(indi_all):
        xi, yi = x[i], y[i]
        # Cell of individual i
        cx_i = int(np.floor(xi / cell_size)) % NC
        cy_i = int(np.floor(yi / cell_size)) % NC

        # Collect all infected neighbours from 3×3 block
        neighbour_js = []
        for dcx in (-1, 0, 1):
            for dcy in (-1, 0, 1):
                flat = ((cx_i + dcx) % NC) * NC + ((cy_i + dcy) % NC)
                if flat in infected_by_cell:
                    neighbour_js.append(infected_by_cell[flat])

        if not neighbour_js:
            continue

        js = np.concatenate(neighbour_js)
        ddx = xi - x[js]
        ddy = yi - y[js]
        # Minimum image convention
        ddx -= L * np.round(ddx / L)
        ddy -= L * np.round(ddy / L)
        r2 = ddx**2 + ddy**2
        mask = r2 <= csl2
        if not mask.any():
            continue
        pi_all[idx] = (sA[js[mask]] * np.exp(-r2[mask] / (2.0 * sig2))).sum()

    rand_vals = rng.random(len(indi_all))
    newly_infected = indi_all[np.floor(pi_all + rand_vals).astype(int) > 0]
    H[newly_infected] = 1
    S[newly_infected] = 0
    return S, H

In [19]:
# =============================================================================
# 4. MAIN
# =============================================================================

if __name__ == "__main__":
    MASTER_SEED = 10
    N     = 10**5
    rho   = 0.5
    L     = np.sqrt(N / rho)
    sig2  = (0.025 * L) ** 2
    rcd   = 50
    ni    = 1
    tspan = 50
    nsim  = 100
    CELL_UPDATE_INTERVAL = 5    # ← modification 1: rebuild every 5 steps

    S_orig = np.zeros(tspan); I_orig = np.zeros(tspan); R_orig = np.zeros(tspan)
    S_cell = np.zeros(tspan); I_cell = np.zeros(tspan); R_cell = np.zeros(tspan)

    csl_fixed = 3.0 * np.sqrt(sig2)

    # ===== ORIGINAL (brute-force) =====
    print(f"Running ORIGINAL method ({nsim} sims)…", flush=True)
    t0 = time.perf_counter()
    rng = np.random.default_rng(MASTER_SEED)
    for sim in range(nsim):
        S, H, R, Hdays, sA, x, y = HealthStatus(N, L, ni, rng)
        for t in range(tspan):
            H, R = InfectedToRecovered(H, R, Hdays, rcd)
            S, H = SusceptibleToInfected_Original(S, H, sA, x, y, sig2, rng)
            Hdays = updateInfectionDays(H, Hdays)
            x, y  = updatePositions(x, y, L, N, sig2, rng)
            S_orig[t] += S.sum(); I_orig[t] += H.sum(); R_orig[t] += R.sum()
    S_orig /= nsim; I_orig /= nsim; R_orig /= nsim
    t_orig = time.perf_counter() - t0
    print(f"  Done in {t_orig:.1f}s")

    # ===== CELL-LISTING (S→I only, update every 5 steps) =====
    print(f"Running CELL-LISTING method ({nsim} sims)…", flush=True)
    t0 = time.perf_counter()
    rng = np.random.default_rng(MASTER_SEED)
    for sim in range(nsim):
        S, H, R, Hdays, sA, x, y = HealthStatus(N, L, ni, rng)
        cell_indices, num_cells, cell_size = build_cell_list(x, y, L, csl_fixed)
        for t in range(tspan):
            # Mod 1: rebuild cell list every 5 steps
            if t % CELL_UPDATE_INTERVAL == 0:
                cell_indices, num_cells, cell_size = build_cell_list(x, y, L, csl_fixed)
            # Mod 2: I→R is shared/identical; cell listing only for S→I
            H, R = InfectedToRecovered(H, R, Hdays, rcd)
            S, H = SusceptibleToInfected_CellList(
                S, H, sA, x, y, sig2, L, rng,
                cell_indices, num_cells, cell_size)
            Hdays = updateInfectionDays(H, Hdays)
            x, y  = updatePositions(x, y, L, N, sig2, rng)
            S_cell[t] += S.sum(); I_cell[t] += H.sum(); R_cell[t] += R.sum()
    S_cell /= nsim; I_cell /= nsim; R_cell /= nsim
    t_cell = time.perf_counter() - t0
    print(f"  Done in {t_cell:.1f}s")
    print(f"  Speed-up: {t_orig/t_cell:.2f}×")

    # ===== PLOT =====
    ttime = np.arange(1, tspan + 1)
    err_S = np.abs(S_orig - S_cell)
    err_I = np.abs(I_orig - I_cell)
    err_R = np.abs(R_orig - R_cell)
    with np.errstate(invalid='ignore', divide='ignore'):
        rel_S = np.where(S_orig > 0, err_S / S_orig * 100, 0.0)
        rel_I = np.where(I_orig > 0, err_I / I_orig * 100, 0.0)
        rel_R = np.where(R_orig > 0, err_R / R_orig * 100, 0.0)

    fig, axes = plt.subplots(3, 1, figsize=(11, 13))
    fig.suptitle(
        "Monte Carlo SIR: Original vs Cell-Listing (S→I only, cell update every 5 steps)\n"
        f"N={N}, nsim={nsim}, seed={MASTER_SEED}  |  "
        f"Original: {t_orig:.1f}s   Cell-list: {t_cell:.1f}s   Speed-up: {t_orig/t_cell:.1f}×",
        fontsize=12, fontweight='bold')

    ax = axes[0]
    ax.plot(ttime, S_orig, 'b-',  lw=2,   label='S  original')
    ax.plot(ttime, S_cell, 'b--', lw=1.5, label='S  cell-list')
    ax.plot(ttime, I_orig, 'r-',  lw=2,   label='I  original')
    ax.plot(ttime, I_cell, 'r--', lw=1.5, label='I  cell-list')
    ax.plot(ttime, R_orig, 'g-',  lw=2,   label='R  original')
    ax.plot(ttime, R_cell, 'g--', lw=1.5, label='R  cell-list')
    ax.set_ylabel('Average count')
    ax.set_title('SIR trajectories  (solid = original, dashed = cell-list)')
    ax.legend(ncol=3, loc='center right'); ax.grid(True, alpha=0.4)

    ax = axes[1]
    ax.plot(ttime, err_S, 'b-',  lw=2, label='|ΔS|')
    ax.plot(ttime, err_I, 'r-.', lw=2, label='|ΔI|')
    ax.plot(ttime, err_R, 'g:',  lw=2, label='|ΔR|')
    ax.set_ylabel('Absolute error (individuals)')
    ax.set_title('Absolute error  |original − cell-list|')
    ax.legend(); ax.grid(True, alpha=0.4)

    ax = axes[2]
    ax.plot(ttime, rel_S, 'b-',  lw=2, label='S (%)')
    ax.plot(ttime, rel_I, 'r-.', lw=2, label='I (%)')
    ax.plot(ttime, rel_R, 'g:',  lw=2, label='R (%)')
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Relative error (%)')
    ax.set_title('Relative error  |original − cell-list| / original × 100 %')
    ax.legend(); ax.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/sir_comparison_v2.png', dpi=150, bbox_inches='tight')
    print("Saved sir_comparison_v2.png")

    print("\n── Summary ────────────────────────────────────────────")
    print(f"  Mean |ΔS| = {err_S.mean():.3f}   Max |ΔS| = {err_S.max():.3f}")
    print(f"  Mean |ΔI| = {err_I.mean():.3f}   Max |ΔI| = {err_I.max():.3f}")
    print(f"  Mean |ΔR| = {err_R.mean():.3f}   Max |ΔR| = {err_R.max():.3f}")
    print(f"  Mean rel S = {rel_S.mean():.2f}%")
    print(f"  Mean rel I = {rel_I.mean():.2f}%")
    print(f"  Mean rel R = {rel_R.mean():.2f}%")

Running ORIGINAL method (100 sims)…


MemoryError: Unable to allocate 4.21 GiB for an array with shape (93988, 6012) and data type float64